# DrugMechDB — Top Metapaths Sankey

Rebuilds the Figure-4 mechanistic-path Sankey with manuscript-style styling:
concept-type colors, abbreviated labels, fixed left-to-right columns, and
source-colored translucent links.

The set of paths shown is **derived** (top-N most frequent metapaths), not
hardcoded — set `TOP_N` below.

## 1. Imports & configuration

In [1]:
import sys
sys.path.insert(0, '/home/agonzalez/data_tools/')

import yaml
import pandas as pd
import plotly.graph_objects as go
from collections import defaultdict
from data_tools.files._analysis import get_metapath_node

DATA  = '/home/agonzalez/DMDB_Analysis/0_data/external/indication_paths.yaml'
OUT   = '/home/agonzalez/DMDB_Analysis/1_code'
TOP_N = 10   # number of most-frequent metapaths to show

## 2. Extract metapaths

For every indication, `get_metapath_node` walks the drug→disease graph and
records each simple path as a concept-type chain
(e.g. `Drug - Protein - BiologicalProcess - Disease`). Duplicates are kept so
that frequencies are preserved.

In [2]:
ind = yaml.safe_load(open(DATA))
all_metapath_nodes = get_metapath_node(ind)

metapaths = [p for id_ in all_metapath_nodes for p in all_metapath_nodes[id_]]
len(metapaths)

5566

## 3. Derive the top-N metapaths

This is what the manuscript notebook does in cells 82–83: rank the metapath
chains by frequency and take the head. With `TOP_N = 10` this reproduces the
original hardcoded `path_interest` list.

In [3]:
metapath_counts = pd.Series(metapaths).value_counts()
path_interest = metapath_counts.head(TOP_N).index.tolist()

print(f"Top {TOP_N} metapaths (of {metapath_counts.size} unique, "
      f"{len(metapaths)} total instances):")
for rank, (mp, n) in enumerate(metapath_counts.head(TOP_N).items(), 1):
    print(f"  {rank:2}. {n:4}  {mp}")

Top 10 metapaths (of 297 unique, 5566 total instances):
   1.  683  Drug - Protein - BiologicalProcess - Disease
   2.  507  Drug - Protein - BiologicalProcess - PhenotypicFeature - Disease
   3.  428  Drug - GeneFamily - BiologicalProcess - OrganismTaxon - Disease
   4.  370  Drug - Protein - BiologicalProcess - OrganismTaxon - Disease
   5.  210  Drug - Protein - ChemicalSubstance - BiologicalProcess - Disease
   6.  171  Drug - Protein - BiologicalProcess - ChemicalSubstance - Disease
   7.  168  Drug - BiologicalProcess - Disease
   8.  168  Drug - Protein - BiologicalProcess - GrossAnatomicalStructure - Disease
   9.  164  Drug - Protein - MolecularActivity - BiologicalProcess - Disease
  10.  135  Drug - ChemicalSubstance - BiologicalProcess - Disease


## 4. Split paths into position columns and count transitions

Each selected chain is split into positions `s1, s2, …`. We then count how many
paths make each adjacent hop (`s1→s2`, `s2→s3`, …); those counts become the
link widths.

In [4]:
# longest selected path sets how many position columns we need (s1..sK)
maxlen = max(len(p.split(" - ")) for p in path_interest)
cols_s = [f's{i}' for i in range(1, maxlen + 1)]

rows = [p.split(" - ") for p in metapaths if p in path_interest]
df = pd.DataFrame(rows, columns=cols_s)
df['index_'] = df.index

# adjacent-position transition counts: s1->s2, s2->s3, ...
list_ = [df.groupby([cols_s[i], cols_s[i + 1]]).agg({'index_': 'count'}).reset_index()
         for i in range(maxlen - 1)]
df.head()

,s1,s2,s3,s4,s5,index_
0,Drug,Protein,BiologicalProcess,Disease,NaN,0
1,Drug,Protein,BiologicalProcess,ChemicalSubstance,Disease,1
2,Drug,Protein,BiologicalProcess,Disease,NaN,2
3,Drug,Protein,BiologicalProcess,Disease,NaN,3
4,Drug,Protein,ChemicalSubstance,BiologicalProcess,Disease,4


## 5. Build nodes and links

A node is a `(conceptType, column)` pair, named like `Protein_M2`. The `_M#`
suffix is the column position, so the same concept type can appear in several
columns. `Disease` is forced into the last column.

In [5]:
LAST = maxlen   # Disease always lands in the last column

names, count_dict, source_list, target_list = [], {}, [], []
for i in range(len(list_)):
    cols = list_[i].columns
    for x, y, z in zip(list_[i][cols[0]], list_[i][cols[1]], list_[i][cols[2]]):
        if x + '_M' + str(i + 1) not in names:
            names.append(x + '_M' + str(i + 1))
        if y == "Disease":
            names.append(y + '_M' + str(LAST))
            source_list.append(x + '_M' + str(i + 1))
            count_dict[x + '_M' + str(i + 1), y + '_M' + str(LAST)] = z
            target_list.append(y + '_M' + str(LAST))
        else:
            count_dict[x + '_M' + str(i + 1), y + '_M' + str(i + 2)] = z
            source_list.append(x + '_M' + str(i + 1))
            target_list.append(y + '_M' + str(i + 2))

## 6. Styling: palette, abbreviations, helpers


In [6]:
palette = {
    'Drug':                     '#E8857B',
    'Protein':                  '#F4D35E',
    'BiologicalProcess':        '#9CCC65',
    'Disease':                  '#B7AFD4',
    'ChemicalSubstance':        '#5C6BC0',
    'PhenotypicFeature':        '#EFA661',
    'OrganismTaxon':            '#4FC3E8',
    'GeneFamily':               '#A1887F',
    'GrossAnatomicalStructure': '#EC6FA0',
    'MolecularActivity':        '#B07CC6',
    'CellularComponent':        '#C5E1A5',
    'Pathway':                  '#E5D986',
    'Cell':                     '#BDBDBD',
    'MacromolecularComplex':    '#FFD180',
}
abbr = {
    'Drug': 'Drug', 'Disease': 'Disease', 'Protein': 'P',
    'BiologicalProcess': 'BP', 'ChemicalSubstance': 'CS',
    'PhenotypicFeature': 'PF', 'OrganismTaxon': 'T', 'GeneFamily': 'G',
    'GrossAnatomicalStructure': 'A', 'MolecularActivity': 'M',
    'CellularComponent': 'CC', 'Pathway': 'PW', 'Cell': 'C',
    'MacromolecularComplex': 'MC',
}

def hex_to_rgba(h, a):
    h = h.lstrip('#')
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return f'rgba({r},{g},{b},{a})'

def ctype(name):       # "Protein_M2" -> "Protein"
    return name.rsplit('_M', 1)[0]

def col(name):         # "Protein_M2" -> 2
    return int(name.rsplit('_M', 1)[1])

## 7. Clean node index and positions



In [7]:
uniq = list(dict.fromkeys(source_list + target_list))
idx = {n: i for i, n in enumerate(uniq)}

node_labels = [abbr.get(ctype(n), ctype(n)) for n in uniq]
node_colors = [palette.get(ctype(n), '#CCCCCC') for n in uniq]

# fixed x by column (evenly spaced); y spread within column, ordered by flow
xpos = {c: 0.02 + (0.96 - 0.02) * (c - 1) / (maxlen - 1) for c in range(1, maxlen + 1)}
by_col = defaultdict(list)
for n in uniq:
    by_col[col(n)].append(n)
flow = defaultdict(float)
for s, t in zip(source_list, target_list):
    flow[s] += count_dict[s, t]
    flow[t] += count_dict[s, t]
ypos = {}
Y_LO, Y_HI = 0.10, 0.90   # keep nodes off the top/bottom edges
for c, members in by_col.items():
    members_sorted = sorted(members, key=lambda n: -flow[n])
    k = len(members_sorted)
    for j, n in enumerate(members_sorted):
        ypos[n] = Y_LO + (Y_HI - Y_LO) * ((j + 0.5) / k)
node_x = [xpos[col(n)] for n in uniq]
node_y = [ypos[n] for n in uniq]

link_src   = [idx[x] for x in source_list]
link_tgt   = [idx[y] for y in target_list]
link_val   = [count_dict[x, y] for x, y in zip(source_list, target_list)]
link_color = [hex_to_rgba(palette.get(ctype(s), '#CCCCCC'), 0.45) for s in source_list]

print('unique nodes:', len(uniq), '| links:', len(link_src), '| total paths:', sum(link_val))

unique nodes: 14 | links: 22 | total paths: 10862


## 8. Build and display the figure

In [8]:
fig = go.Figure(data=[go.Sankey(
    arrangement='fixed',
    node=dict(
        pad=12,
        thickness=14,
        line=dict(color='white', width=0.8),
        label=node_labels,
        color=node_colors,
        x=node_x,
        y=node_y,
        hovertemplate='%{label}<br>%{value} paths<extra></extra>',
    ),
    link=dict(
        source=link_src,
        target=link_tgt,
        value=link_val,
        color=link_color,
        hovertemplate='%{source.label} \u2192 %{target.label}<br>%{value} paths<extra></extra>',
    ),
)])
fig.update_layout(
    font=dict(family='Arial', size=12, color='#222'),
    paper_bgcolor='white', plot_bgcolor='white',
    width=760, height=420, margin=dict(l=45, r=80, t=20, b=20),
)
fig.show()

## 9. Export (PNG / SVG / interactive HTML)

In [9]:
# display stays small (cell above); export full-size for the manuscript
fig.write_image(OUT + '/mp_sankey_improved.png', width=1320, height=760, scale=2)
fig.write_image(OUT + '/mp_sankey_improved.svg', width=1320, height=760)
fig.write_html(OUT + '/mp_sankey_improved.html')
print('wrote mp_sankey_improved.{png,svg,html}')

/tmp/claude-142609/ipykernel_3755623/1895289669.py:2: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(OUT + '/mp_sankey_improved.png', width=1320, height=760, scale=2)


wrote mp_sankey_improved.{png,svg,html}


/tmp/claude-142609/ipykernel_3755623/1895289669.py:3: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(OUT + '/mp_sankey_improved.svg', width=1320, height=760)
